# Module 2: What Exactly is an RDD?

Objective: By the end of this chapter, students should not only know the definition of an RDD but also understand why every word in RDD exists, how Spark uses it internally, and how it differs from normal collections.


numbers = [10, 20, 30, 40, 50]

Where is this data stored?

Inside your computer’s RAM.

Everything happens inside one machine.

RDD
↓
Resilient Distributed Dataset

What Does “Resilient” Mean?

Resilient means--> Able to recover from failure.

### Why is RDD Immutable?

Because immutability:

- Makes fault recovery easier.
- Avoids conflicts when many tasks run in parallel.
- Simplifies optimization and lineage tracking.

### Advantages of RDD

- Fault tolerant
- Distributed processing
- Parallel execution
- Scalable to many machines
- Supports lazy evaluation
- Works well for low-level transformations and custom processing

### Limitations of RDD

RDDs also have drawbacks.

- No automatic query optimization.
- No schema information.
- More verbose code.
- DataFrames and Datasets are generally preferred for structured data because Spark can optimize them better.


### Full Internal Flow

Python Program
↓
SparkSession
↓
SparkContext
↓
Driver
↓
RDD Created
↓
Partitions Planned
↓
(No execution yet)
↓
collect()
↓
Job
↓
Stage
↓
Task
↓
Executor
↓
Result
↓
Driver


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RDD Fundamentals")
    .master("local[2]")
    .getOrCreate()
)

sc = spark.sparkContext

print(sc.appName)
print(sc.uiWebUrl)

In [ ]:
numbers=[10,20,30,40,50]

In [ ]:
rdd=sc.parallelize(numbers)
print("RDD is Created")

In [ ]:
rdd.collect()

In [ ]:
spark.stop()

# Creating RDD 

In [ ]:
spark.stop()

In [ ]:
from pyspark.sql import SparkSession
spark=(
    SparkSession.builder
    .appName("RDD Fundamentals")
    .master("local[*]")
    .getOrCreate()
)
sc=spark.sparkContext
print(sc.uiWebUrl)

sc

### Method 1

sc.parallelize() - 

In [ ]:
numbers=[1,2,3,4,5,6,7,8,9,10]

print(type(numbers))

In [ ]:
rdd = sc.parallelize(numbers,3)

In [ ]:
print(type(rdd))

In [ ]:
rdd.getNumPartitions()

In [ ]:
rdd.glom().collect()

In [ ]:
rdd.collect()

In [ ]:
rdd1=sc.parallelize(range(1,11),5)
print(rdd1.glom().collect())

### Method 2 : sc.textFile()



In [ ]:
employee_rdd=sc.textFile("employee.txt")

In [ ]:
employee_rdd.collect()

In [ ]:
employee_rdd.getNumPartitions()

In [ ]:
employee_rdd.glom().collect()

# RDD Partitions 

### What is Partition?

- A Logical chucnk of an RDD that can be processed independently 

- RDD =[1,2,3,4,5,6,7,8] - 2 PARTITIONS 

RDD 
|_ PARTITION 0 [1,2,3]
|
|- PARTITION 1 [8,8,9]

### Why Does Spark needs Partitions 

- To distribute the data across multiple nodes in a cluster for parallel processing.




In [ ]:
numbers = list(range(1,21))

In [ ]:
numbers

In [ ]:
rdd=sc.parallelize(numbers,4)

In [ ]:
# Check partition count 

rdd.getNumPartitions()

In [ ]:
# Actual Contents inside partition 

rdd.glom().collect()

# What does Glom()() do?    
# Glom() converts each partition into a list and returns an RDD of lists.



# Maximum parallel tasks at one time ---> Total avilable cores in the cluster. 

### Too Few Partition 
- Underutilization of resources.

### Too Many Partitions 
- Overhead of managing too many small tasks.

- Problems
- Task scheduling overhead 
- too much metadata 
- Excessive task launch cost 
- Possible many tiny output files 
- Driver schdeduling pressre 

### What Determines Partition Count 

- Number of cores in the cluster

- For sc.textFile() partition size depends on:

- input files 
- file sizes 
- Hadppod input slpits 
- clock/split settings 
- compression 
- filesystem 
- Spark Configuration 






In [ ]:
spark.stop()

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TooManyPartitions")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark Version:", sc.version)
print(sc.appName)
print(sc.uiWebUrl)
print("Available Cores:", sc.defaultParallelism)

In [ ]:
rdd_normal=sc.parallelize(range(1_000_000),8)
print("Number of Partitions:", rdd_normal.getNumPartitions())

In [ ]:
import time 
start_time=time.time()
result=rdd_normal.map(lambda x:x*2).sum()
end_time=time.time()
print("Result:", result)
print("Time taken:", end_time - start_time)

In [ ]:
rdd_many=sc.parallelize(range(1_000_000),10000)
print("Number of Partitions:", rdd_many.getNumPartitions())

In [ ]:
import time 
start_time=time.time()
result=rdd_many.map(lambda x:x*2).sum()
end_time=time.time()
print("Result:", result)
print("Time taken:", end_time - start_time)

In [ ]:
import time
partition_count = [
    2,
    4,
    8,
    16,
    100,
    1000,
    5000
]

for partitions in partition_count:
    rdd=sc.parallelize(range(1_000_000),partitions)
    start_time=time.time()
    result=(
        rdd.map(lambda x:x*2).filter(lambda x:x%3==0).sum()
    )
    end_time=time.time()
    print(
        f"Partitions:{partitions:<5}"
        f"Time: {end_time-start_time:.4f} seconds"
    )

In [ ]:
rdd=sc.parallelize(range(1,13),4)

In [ ]:
def show_partitions(index,iterator):
    for value in iterator:
        yield(index,value)
result=rdd.mapPartitionsWithIndex(show_partitions)
result.collect()


# RDD Lazy Evalaution and Lineage 

- Transformations such as map() and filter() do not execute immediately. Spark remembers them and waits until an action is called.

### What is Lazy Evaluation

- Spark delays execution of transformations until it actually needs a result.

- When you write them, Spark builds lineage/dependencies.

- It doesn’t immediately run the entire computation.


rdd = sc.parallelize([1, 2, 3, 4, 5])

result = rdd.map(lambda x: print("Processing:", x))

Output :

Processing: 1
Processing: 2
Processing: 3
Processing: 4
Processing: 5

- But simply defining the transformation does not require Spark to process all records.

- result.collect()

Now the task actually executes.

- In local mode, you’ll see task-side output depending on the notebook/log environment.

map()
   ↓
No action
   ↓
No full Spark job

collect()
   ↓
Action
   ↓
Job starts

# Transformation vs Action

### Transformation

- Takes one RDD and produces another RDD.

RDD1
 ↓
map
 ↓
RDD2

- Transformations are normally lazy. 

### Action

- Produces a final result or writes output.

RDD
 ↓
Action
 ↓
Spark Job









RDD transformations create a **new RDD from an existing RDD**.

> **Important:** Transformations are **lazy**. Spark does not execute them immediately. Execution starts when an **Action** is called.

---

## Complete RDD Transformation List

```text
RDD TRANSFORMATIONS
│
├── 1. BASIC / ELEMENT TRANSFORMATIONS
│   │
│   ├── map()
│   ├── flatMap()
│   ├── filter()
│   ├── mapPartitions()
│   ├── mapPartitionsWithIndex()
│   ├── glom()
│   ├── sample()
│   ├── pipe()
│   └── keyBy()
│
├── 2. SET / DATASET TRANSFORMATIONS
│   │
│   ├── union()
│   ├── distinct()
│   ├── intersection()
│   ├── subtract()
│   └── cartesian()
│
├── 3. PAIR RDD — AGGREGATION TRANSFORMATIONS
│   │
│   ├── reduceByKey()
│   ├── groupByKey()
│   ├── aggregateByKey()
│   ├── combineByKey()
│   ├── foldByKey()
│   └── groupBy()
│
├── 4. PAIR RDD — VALUE TRANSFORMATIONS
│   │
│   ├── mapValues()
│   ├── flatMapValues()
│   ├── keys()
│   └── values()
│
├── 5. SORTING TRANSFORMATIONS
│   │
│   ├── sortByKey()
│   └── sortBy()
│
├── 6. JOIN / COGROUP TRANSFORMATIONS
│   │
│   ├── join()
│   ├── leftOuterJoin()
│   ├── rightOuterJoin()
│   ├── fullOuterJoin()
│   ├── cogroup()
│   └── groupWith()
│
├── 7. PARTITION TRANSFORMATIONS
│   │
│   ├── partitionBy()
│   ├── repartition()
│   └── coalesce()
│
└── 8. OTHER / SPECIAL TRANSFORMATIONS
    │
    ├── zip()
    ├── zipWithIndex()
    ├── zipWithUniqueId()
    ├── randomSplit()
    └── repartitionAndSortWithinPartitions()

# RDD Transformations — Narrow vs Wide

RDD transformations are mainly classified into **Narrow Transformations** and **Wide Transformations** based on the dependency between partitions.

---

## Narrow Transformations vs  Wide Transformations

```text
RDD TRANSFORMATIONS
│
├── NARROW TRANSFORMATIONS
│   │
│   │   Child partition depends on limited parent partition(s)
│   │   Normally NO SHUFFLE
│   │
│   ├── BASIC / ELEMENT
│   │   ├── map()
│   │   ├── flatMap()
│   │   ├── filter()
│   │   ├── mapPartitions()
│   │   ├── mapPartitionsWithIndex()
│   │   ├── glom()
│   │   ├── sample()
│   │   ├── pipe()
│   │   └── keyBy()
│   │
│   ├── SET / DATASET
│   │   ├── union()
│   │   └── cartesian()                  ← Special case
│   │
│   ├── PAIR RDD / VALUE
│   │   ├── mapValues()
│   │   ├── flatMapValues()
│   │   ├── keys()
│   │   └── values()
│   │
│   ├── PARTITION
│   │   └── coalesce()                   ← shuffle=False
│   │
│   └── OTHER / SPECIAL
│       ├── zip()
│       ├── zipWithIndex()               ← Special case
│       ├── zipWithUniqueId()
│       └── randomSplit()
│
│
└── WIDE TRANSFORMATIONS
    │
    │   Data generally needs redistribution across partitions
    │   SHUFFLE is normally required
    │
    ├── SET / DATASET
    │   ├── distinct()
    │   ├── intersection()
    │   └── subtract()
    │
    ├── AGGREGATION
    │   ├── groupBy()
    │   ├── reduceByKey()
    │   ├── groupByKey()
    │   ├── aggregateByKey()
    │   ├── combineByKey()
    │   └── foldByKey()
    │
    ├── SORTING
    │   ├── sortByKey()
    │   └── sortBy()
    │
    ├── JOIN / COGROUP
    │   ├── join()
    │   ├── leftOuterJoin()
    │   ├── rightOuterJoin()
    │   ├── fullOuterJoin()
    │   ├── cogroup()
    │   └── groupWith()
    │
    ├── PARTITION
    │   ├── partitionBy()
    │   ├── repartition()
    │   └── coalesce(shuffle=True)
    │
    └── OTHER
        └── repartitionAndSortWithinPartitions()
```

---

## Rule

### Narrow Transformation

**One child partition depends on a limited number of parent partitions.**

```text
Parent RDD                 Child RDD

Partition 1  ────────────► Partition 1

Partition 2  ────────────► Partition 2

Partition 3  ────────────► Partition 3

                No Shuffle
```

Examples:

- `map()`
- `flatMap()`
- `filter()`
- `mapPartitions()`
- `mapValues()`

---

### Wide Transformation

**A child partition may require data from multiple parent partitions.**

```text
Parent RDD                      Child RDD

Partition 1 ─────┬────────────► Partition 1
                 │
Partition 2 ─────┼────────────► Partition 2
                 │
Partition 3 ─────┴────────────► Partition 3

                  SHUFFLE
```

Examples:

- `reduceByKey()`
- `groupByKey()`
- `distinct()`
- `sortByKey()`
- `repartition()`
- `join()`

---

## Key Difference

| Narrow Transformation         | Wide Transformation |
|---|---|
| Normally no shuffle           | Normally causes shuffle |
| Data stays locally accessible | Data may move across partitions/executors |
| Faster | More expensive       |
| Usually stays in same stage   | Creates a stage boundary |
| Less network I/O              | More network I/O |
| Example: `map()`              | Example: `reduceByKey()` |

---

## Remember

```text
NARROW
   │
   ├── No Shuffle
   │
   ├── Less Network I/O
   │
   ├── Faster
   │
   └── Same Stage
           

WIDE
   │
   ├── Shuffle
   │
   ├── Network I/O
   │
   ├── More Expensive
   │
   └── New Stage Boundary
```

> **Golden Rule:**  
> **Narrow → No Shuffle → Same Stage**  
> **Wide → Shuffle → Stage Boundary**

In [ ]:
numbers=[1,2,3,4,5]
result=[x *2 for x in numbers]
print(result)

In [ ]:
rdd=sc.parallelize([1,2,3,4,5])
result=rdd.map(lambda x: print("Processing",x))

In [ ]:
result.collect()

## Transformation vs Action 

- Transformation: Takes one RDD and produces another RDD 

- rdd2=rdd1.map(lambda x:x*2)

- Transformations are lazy 

## Action 

- Produces a final result or write a output 
- rdd.collect(), rdd.count(), rdd.first(), rdd.take(),rdd.reduce()

In [ ]:
rdd1=sc.parallelize(range(1,11))
rdd2=rdd1.map(lambda x:x*2)
rdd3=rdd2.filter(lambda x:x>10)
rdd4=rdd3.map(lambda x:(x,x**2))

In [ ]:
rdd4.collect()

In [ ]:
rdd1=sc.parallelize(range(1,11),2)
rdd2=rdd1.map(lambda x:x*2)
rdd3=rdd2.filter(lambda x:x>10)

In [ ]:
print(rdd3.toDebugString())

b'(2) PythonRDD[5] at RDD at PythonRDD.scala:53 []\n |  
ParallelCollectionRDD[4] at readRDDFromFile at PythonRDD.scala:289 []'

In [ ]:
print("rdd1")
print(rdd1.toDebugString())

print("\nrdd2")
print(rdd2.toDebugString())

print("\nrdd3")
print(rdd3.toDebugString())

In [ ]:
print(rdd1.glom().collect())

In [ ]:
print(rdd2.glom().collect())

In [ ]:
print(rdd3.glom().collect())

# Narrow and Wide Transformation 



In [ ]:
rdd1=sc.parallelize([1,2,3,4,5,6,7,8],2) # Parent RDD 
rdd2=rdd1.map(lambda x:x*2) # Child RDD 

# For Creating one partiton of RDD2, How many partition of RDD1 are needed 

## Narrow Transformation 

- Each child partition depends on only a small/fixed number of parent partition 

- No Shullfle 
